In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import unicodedata
import re

import pandas as pd
import plotly.express as px
import plotly.io as pio

if "vscode" in pio.renderers:
    pio.renderers.default = "vscode"
else:
    pio.renderers.default = "notebook_connected"

try:
    import geobr
except ImportError as exc:
    raise ImportError("Pacote 'geobr' não encontrado. Instale com: pip install geobr") from exc


In [ ]:
# Caminhos de entrada
CSV_MUNICIPIOS = Path("/home/anna/tcc/Tratamento/Municípios/tabela_municipios_regionais.csv")
CSV_POPULACAO_FAIXA_ETARIA = Path("/home/anna/tcc/Tratamento/População - Faixa Etária/tabela_populacao_faixa_etaria.csv")

# Parâmetros
ANO_POP = 2022  # ano usado nos mapas/gráficos populacionais
FAIXAS_DECENAIS = ["0-9", "10-19", "20-29", "30-39", "40-49", "50-59", "60-69", "70 ou mais"]

# (Opcional) Pirâmide etária feminina (precisa de um arquivo com sexo + faixas de 5 anos)
# Esperado: colunas [faixa_5anos, pop_feminina] ou similar (ajustável na Célula 5).
CSV_PIRAMIDE_FEMININA_5ANOS = None  # ex.: Path("/caminho/para/piramide_feminina.csv")

In [ ]:
# 3) Carregamento + preparação dos dados (municípios, regionais, população)
import json
import geopandas as gpd

def _norm_txt(txt: str) -> str:
    txt = "" if pd.isna(txt) else str(txt)
    txt = unicodedata.normalize("NFKD", txt).encode("ascii", "ignore").decode("ascii")
    return re.sub(r"\s+", " ", txt).strip().lower()


def _as_ibge7(val) -> str:
    """Normaliza código IBGE de município para 7 dígitos (lida com floats tipo '3100104.0')."""
    if pd.isna(val):
        return ""
    s = str(val).strip()
    # Caso comum: veio como float serializado ('3100104.0')
    if re.fullmatch(r"\d+(?:\.0+)?", s):
        s = s.split(".", 1)[0]
    s = re.sub(r"\D", "", s)
    if len(s) > 7:
        # Evita casos do tipo '31001040' (extra 0 do '.0')
        s = s[:7]
    return s.zfill(7)


if not CSV_MUNICIPIOS.exists():
    raise FileNotFoundError(f"Não encontrei: {CSV_MUNICIPIOS}")
if not CSV_POPULACAO_FAIXA_ETARIA.exists():
    raise FileNotFoundError(f"Não encontrei: {CSV_POPULACAO_FAIXA_ETARIA}")

# Regionais por município
df_reg = pd.read_csv(CSV_MUNICIPIOS, dtype={"codigo_ibge": "string"})
df_reg = df_reg[["codigo_ibge", "municipio_normalizado", "regional_original", "regional_normalizada"]].copy()
df_reg["codigo_ibge"] = df_reg["codigo_ibge"].map(_as_ibge7)

# População por faixa etária (decenal)
df_pop = pd.read_csv(CSV_POPULACAO_FAIXA_ETARIA)
df_pop = df_pop.rename(columns={"CÓDIGO": "codigo_ibge", "MUNICÍPIOS": "municipio"}).copy()
df_pop["codigo_ibge"] = df_pop["codigo_ibge"].map(_as_ibge7)
df_pop["municipio"] = df_pop["municipio"].astype(str).map(_norm_txt)

cols_ano = [f"{ANO_POP}_{fx}" for fx in FAIXAS_DECENAIS]
missing = [c for c in cols_ano if c not in df_pop.columns]
if missing:
    raise KeyError(f"CSV de população não tem as colunas esperadas para {ANO_POP}: {missing}")

# Faixa predominante por município (ano ANO_POP)
pop_vals = df_pop[cols_ano].apply(pd.to_numeric, errors="coerce").fillna(0)
pred_col = pop_vals.idxmax(axis=1)  # ex.: '2022_40-49'
df_pop["faixa_predominante"] = pred_col.astype(str).str.replace(f"{ANO_POP}_", "", regex=False)
df_pop["faixa_predominante"] = df_pop["faixa_predominante"].where(pop_vals.sum(axis=1) > 0)

# Geometria dos municípios de MG
try:
    gdf_mg = geobr.read_municipality(code_muni="MG", year=2020)
except TypeError:
    gdf_mg = geobr.read_municipality(code_muni="MG")

gdf_mg = gdf_mg.to_crs(4326)
code_col = "code_muni" if "code_muni" in gdf_mg.columns else None
if not code_col:
    raise KeyError(f"Não encontrei coluna 'code_muni' no geobr. Colunas: {list(gdf_mg.columns)}")

gdf_mg["codigo_ibge"] = gdf_mg[code_col].map(_as_ibge7)

# Junta atributos (regional + população)
gdf = gdf_mg.merge(df_reg, on="codigo_ibge", how="left")
gdf = gdf.merge(df_pop[["codigo_ibge", "faixa_predominante"]], on="codigo_ibge", how="left")

print("Municípios (geometria):", len(gdf_mg))
print("Com regional:", int(gdf["regional_original"].notna().sum()))
print("Com faixa predominante:", int(gdf["faixa_predominante"].notna().sum()))

# GeoJSON para Plotly (sem Mapbox)
geojson_mg = json.loads(
    gdf[["codigo_ibge", "geometry", "regional_original", "faixa_predominante"]].to_json()
 )

In [ ]:
# 4) Mapas (formato MG)
# Gráfico 1 — Municípios de Minas Gerais (apenas contorno/polígono)
fig1 = px.choropleth(
    gdf.assign(_dummy=1),
    geojson=geojson_mg,
    locations="codigo_ibge",
    featureidkey="properties.codigo_ibge",
    color="_dummy",
    color_continuous_scale=["#2D9CDB", "#2D9CDB"],
    title="Gráfico 1 — Municípios de Minas Gerais",
)
fig1.update_traces(marker_line_color="white", marker_line_width=0.3)
fig1.update_layout(coloraxis_showscale=False)
fig1.update_geos(fitbounds="locations", visible=False)
fig1.show()

# Gráfico 2 — Regionais de saúde (cada regional 1 cor)
gdf_reg = gdf.dropna(subset=["regional_original"]).copy()
fig2 = px.choropleth(
    gdf_reg,
    geojson=geojson_mg,
    locations="codigo_ibge",
    featureidkey="properties.codigo_ibge",
    color="regional_original",
    title="Gráfico 2 — Regionais de Saúde (MG)",
)
fig2.update_traces(marker_line_color="white", marker_line_width=0.3)
fig2.update_geos(fitbounds="locations", visible=False)
fig2.show()

# Gráfico 3 — População por município (cor = faixa etária predominante no município)
gdf_pred = gdf.dropna(subset=["faixa_predominante"]).copy()
ordem_faixas = FAIXAS_DECENAIS[:]
gdf_pred["faixa_predominante"] = pd.Categorical(
    gdf_pred["faixa_predominante"], categories=ordem_faixas, ordered=True
 )
fig3 = px.choropleth(
    gdf_pred,
    geojson=geojson_mg,
    locations="codigo_ibge",
    featureidkey="properties.codigo_ibge",
    color="faixa_predominante",
    category_orders={"faixa_predominante": ordem_faixas},
    title=f"Gráfico 3 — Faixa etária predominante por município (MG, {ANO_POP})",
)
fig3.update_traces(marker_line_color="white", marker_line_width=0.3)
fig3.update_geos(fitbounds="locations", visible=False)
fig3.show()

In [ ]:
# 5) Gráfico 4 para todos os anos disponíveis + Pirâmide etária (feminina)

ANOS_DESEJADOS = [1970, 1980, 1991, 2000, 2010, 2022]

# Gráfico 4 — Faixa etária no estado (soma MG) para cada ano
for ano in ANOS_DESEJADOS:
    cols_ano = [f"{ano}_{fx}" for fx in FAIXAS_DECENAIS]
    faltantes = [c for c in cols_ano if c not in df_pop.columns]
    if faltantes:
        print(f"[Aviso] Ano {ano} sem todas as colunas esperadas. Faltando: {faltantes}")
        continue

    pop_estado = df_pop[cols_ano].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=0)
    pop_estado.index = [c.replace(f"{ano}_", "") for c in pop_estado.index]

    df_estado = pop_estado.reindex(FAIXAS_DECENAIS).reset_index()
    df_estado.columns = ["faixa_etaria", "populacao"]

    fig4 = px.bar(
        df_estado,
        x="populacao",
        y="faixa_etaria",
        orientation="h",
        text="populacao",
        title=f"Gráfico 4 — População por faixa etária no estado (MG, {ano})",
    )
    fig4.update_traces(texttemplate="%{text:,}", textposition="outside")
    fig4.update_layout(xaxis_title="População", yaxis_title="Faixa etária")
    fig4.show()

# Pirâmide etária — População feminina (5 em 5 anos)
if CSV_PIRAMIDE_FEMININA_5ANOS is None:
    print(
        "\n[Pendência] Para a pirâmide etária feminina (5 em 5 anos), preciso de um CSV com população feminina por faixa de 5 anos.\n"
        "Exemplo esperado: colunas 'faixa_5anos' (0-4, 5-9, ..., 100+) e 'pop_feminina'.\n"
        "Defina CSV_PIRAMIDE_FEMININA_5ANOS na Célula 2 e rode novamente."
    )
else:
    df_pir = pd.read_csv(CSV_PIRAMIDE_FEMININA_5ANOS)
    col_faixa = "faixa_5anos" if "faixa_5anos" in df_pir.columns else None
    col_pop = "pop_feminina" if "pop_feminina" in df_pir.columns else None
    if not col_faixa or not col_pop:
        raise KeyError("CSV_PIRAMIDE_FEMININA_5ANOS precisa ter colunas 'faixa_5anos' e 'pop_feminina'.")

    df_pir = df_pir[[col_faixa, col_pop]].copy()
    df_pir[col_pop] = pd.to_numeric(df_pir[col_pop], errors="coerce").fillna(0)
    df_pir = df_pir[df_pir[col_pop] > 0]
    if df_pir.empty:
        raise ValueError("Sem dados > 0 para pirâmide etária feminina.")

    def _ordem_faixa_5a(x: str):
        s = str(x).strip().lower()
        m = re.match(r"^(\d+)", s)
        return int(m.group(1)) if m else 10**9

    df_pir = df_pir.sort_values(col_faixa, key=lambda s: s.map(_ordem_faixa_5a))

    fig_pir = px.bar(
        df_pir,
        x=col_pop,
        y=col_faixa,
        orientation="h",
        text=col_pop,
        title="Pirâmide etária — População feminina (MG)",
    )
    fig_pir.update_traces(texttemplate="%{text:,}", textposition="outside")
    fig_pir.update_layout(xaxis_title="População feminina", yaxis_title="Faixa etária (5 anos)")
    fig_pir.show()